In [1]:
from pathlib import Path
import re

import pandas as pd

try:
    display
except NameError:
    def display(obj):
        print(obj)

## 1. Khai bao duong dan

In [2]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_CSV = PROJECT_ROOT / "data" / "processed" / "student_voice_merged.csv"
OUTPUT_CSV = PROJECT_ROOT / "data" / "processed" / "student_voice_enriched.csv"
REPORT_PATH = PROJECT_ROOT / "outputs" / "reports" / "label_enrichment_report.md"

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_CSV)
print("Output:", OUTPUT_CSV)
print("Report:", REPORT_PATH)

Input: d:\AI thuc chien\Student Voice Intelligence\data\processed\student_voice_merged.csv
Output: d:\AI thuc chien\Student Voice Intelligence\data\processed\student_voice_enriched.csv
Report: d:\AI thuc chien\Student Voice Intelligence\outputs\reports\label_enrichment_report.md


## 2. Load data va validate schema


In [3]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Khong tim thay {INPUT_CSV}. Hay chay Phase 1 truoc.")

df = pd.read_csv(INPUT_CSV)

REQUIRED_COLUMNS = [
    "text",
    "source_dataset",
    "split_original",
    "sentiment_raw",
    "sentiment_std_code",
    "sentiment_std_3class",
    "sentiment_std_4class",
    "topic_std",
    "is_toxic",
    "urgency_level",
]

missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing:
    raise ValueError(f"File input thieu cot: {missing}")

df["text"] = df["text"].fillna("").astype(str)

print("Shape:", df.shape)
display(df.head())

Shape: (49141, 11)


,text,source_dataset,split_original,sentiment_raw,sentiment_std_code,sentiment_std_3class,sentiment_std_4class,topic_raw,topic_std,is_toxic,urgency_level
0,em xin chào các anh chị em em là sinh viên vừa...,NEU_ESC,train,0,0,neutral,neutral,2,academic,0,low
1,xây dựng mô hình sư phạm chuẩn mực sáng tạo ti...,NEU_ESC,train,0,0,neutral,neutral,2,academic,0,low
2,sao lại ghét cái kiểu làm sai xong khóc lóc ăn...,NEU_ESC,train,2,2,negative,negative,3,others,0,low
3,chào thầy đăng ký học ghép . môn học hiện hệ t...,NEU_ESC,train,0,0,neutral,neutral,2,academic,0,low
4,con bé vẫn hoang mang lắm .,NEU_ESC,train,2,2,negative,negative,3,others,0,low


## 3. Rule-based toxic labeling


In [4]:
TOXIC_PATTERNS = {
    "profanity_strong": [
        r"\bđịt\b", r"\bdjt\b", r"\bđm\b", r"\bdm\b", r"\bđcm\b", r"\bdcm\b",
        r"\blồn\b", r"\blon\b", r"\bcặc\b", r"\bcac\b", r"\bđéo\b", r"\bdeo\b",
    ],
    "insult": [
        r"\bngu\b", r"\bngu dốt\b", r"\bóc chó\b", r"\boc cho\b", r"\bchó chết\b",
        r"\bham lon\b", r"\bhãm\b", r"\bkhốn nạn\b", r"\bmat day\b", r"\bmất dạy\b",
    ],
    "sexual_slur": [
        r"\bđĩ\b", r"\bdi~\b", r"\bphò\b", r"\bpho\b",
    ],
    "internet_slang": [
        r"\bvcl\b", r"\bvl\b", r"\bclm\b", r"\bclgt\b",
    ],
}

def normalize_for_rule(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def find_pattern_hits(text, pattern_groups):
    normalized = normalize_for_rule(text)
    hits = []
    for group_name, patterns in pattern_groups.items():
        for pattern in patterns:
            if re.search(pattern, normalized, flags=re.IGNORECASE):
                hits.append(group_name)
                break
    return sorted(set(hits))

df["is_toxic_seed"] = df["is_toxic"].astype(int)
df["toxic_rule_hits"] = df["text"].apply(lambda text: ",".join(find_pattern_hits(text, TOXIC_PATTERNS)))
df["is_toxic_rule"] = (df["toxic_rule_hits"].str.len() > 0).astype(int)

# Final toxic = toxic co san trong NEU hoac toxic bat duoc bang rule.
df["is_toxic"] = ((df["is_toxic_seed"] == 1) | (df["is_toxic_rule"] == 1)).astype(int)

def toxic_source(row):
    if row["is_toxic_seed"] == 1 and row["is_toxic_rule"] == 1:
        return "seed_and_rule"
    if row["is_toxic_seed"] == 1:
        return "seed_neu"
    if row["is_toxic_rule"] == 1:
        return "rule"
    return "default_non_toxic"

df["toxic_label_source"] = df.apply(toxic_source, axis=1)

display(pd.crosstab(df["source_dataset"], df["is_toxic"], margins=True))
display(df["toxic_label_source"].value_counts().to_frame("count"))

is_toxic,0,1,All
source_dataset,,,
NEU_ESC,31835,1131,32966
UIT_VSFC,16174,1,16175
All,48009,1132,49141


,count
toxic_label_source,
default_non_toxic,48009
seed_and_rule,497
seed_neu,348
rule,287


## 4. Rule-based urgency labeling


In [5]:
URGENCY_HIGH_PATTERNS = {
    "safety": [
        r"\bmất an toàn\b", r"\bmat an toan\b", r"\bnguy hiểm\b", r"\bnguy hiem\b",
        r"\btai nạn\b", r"\btai nan\b", r"\bchấn thương\b", r"\bchan thuong\b", r"\bcấp cứu\b", r"\bcap cuu\b",
        r"\bcháy\b", r"\bro điện\b", r"\brò điện\b", r"\bđiện giật\b", r"\bdien giat\b",
        r"\bnổ\s+(điện|dien|gas|bình|binh|lớn|lon)\b",
    ],
    "violence_harassment": [
        r"bạo lực", r"bao luc", r"đe dọa", r"de doa", r"quấy rối", r"quay roi",
        r"xâm hại", r"xam hai", r"gạ gẫm", r"ga gam",
    ],
    "blocked_learning": [
        r"không thể học", r"khong the hoc", r"không thể thi", r"khong the thi",
        r"không vào được lớp", r"khong vao duoc lop",
    ],
}

URGENCY_MEDIUM_PATTERNS = {
    "technical_issue": [
        r"wifi", r"mạng yếu", r"mang yeu", r"mất mạng", r"mat mang", r"hệ thống lỗi", r"he thong loi",
        r"server", r"đăng ký lỗi", r"dang ky loi", r"không đăng ký được", r"khong dang ky duoc",
    ],
    "facility_issue": [
        r"máy chiếu", r"may chieu", r"điều hòa", r"dieu hoa", r"phòng nóng", r"phong nong",
        r"thiết bị hỏng", r"thiet bi hong", r"phòng quá tải", r"phong qua tai",
    ],
    "admin_issue": [
        r"lịch học", r"lich hoc", r"lịch thi", r"lich thi", r"trùng lịch", r"trung lich",
        r"chậm phản hồi", r"cham phan hoi", r"không phản hồi", r"khong phan hoi",
    ],
}

def label_urgency(text):
    high_hits = find_pattern_hits(text, URGENCY_HIGH_PATTERNS)
    if high_hits:
        return "high", ",".join(high_hits), "rule_high"

    medium_hits = find_pattern_hits(text, URGENCY_MEDIUM_PATTERNS)
    if medium_hits:
        return "medium", ",".join(medium_hits), "rule_medium"

    return "low", "", "default_low"

urgency_result = df["text"].apply(label_urgency)
df["urgency_level"] = urgency_result.apply(lambda item: item[0])
df["urgency_rule_hits"] = urgency_result.apply(lambda item: item[1])
df["urgency_label_source"] = urgency_result.apply(lambda item: item[2])

display(df["urgency_level"].value_counts().to_frame("count"))
display(pd.crosstab(df["source_dataset"], df["urgency_level"], margins=True))

,count
urgency_level,
low,48502
medium,445
high,194


urgency_level,high,low,medium,All
source_dataset,,,,
NEU_ESC,192,32531,243,32966
UIT_VSFC,2,15971,202,16175
All,194,48502,445,49141


## 5. Topic group taxonomy


In [6]:
TOPIC_GROUP_MAP = {
    "academic": "teaching_learning",
    "lecturer": "teaching_learning",
    "training_program": "teaching_learning",
    "facilities": "facilities",
    "student_services": "student_services",
    "help_share": "student_services",
    "jobs_recruitment": "career_jobs",
    "news": "events_news",
    "clubs_events": "events_news",
    "personal_affairs": "personal_social",
    "social_affairs": "personal_social",
    "spam": "spam",
    "others": "others",
}

df["topic_group"] = df["topic_std"].map(TOPIC_GROUP_MAP)

if df["topic_group"].isna().any():
    missing_topics = sorted(df.loc[df["topic_group"].isna(), "topic_std"].dropna().unique().tolist())
    raise ValueError(f"Co topic_std chua map sang topic_group: {missing_topics}")

display(df["topic_group"].value_counts().to_frame("count"))
display(pd.crosstab(df["source_dataset"], df["topic_group"], margins=True))

,count
topic_group,
teaching_learning,25159
others,15218
student_services,3028
personal_social,2247
events_news,1564
career_jobs,808
facilities,712
spam,405


topic_group,career_jobs,events_news,facilities,others,personal_social,spam,student_services,teaching_learning,All
source_dataset,,,,,,,,,
NEU_ESC,808,1564,0,14402,2247,405,3028,10512,32966
UIT_VSFC,0,0,712,816,0,0,0,14647,16175
All,808,1564,712,15218,2247,405,3028,25159,49141


## 6. Danh dau mau can review


In [7]:
df["label_needs_review"] = (
    (df["toxic_label_source"] == "rule")
    | (df["urgency_label_source"].isin(["rule_high", "rule_medium"]))
).astype(int)

display(df["label_needs_review"].value_counts().to_frame("count"))

,count
label_needs_review,
0,48220
1,921


## 7. Audit nhanh cac label moi


In [8]:
print("Rule-added toxic examples:")
display(
    df[(df["toxic_label_source"] == "rule")][
        ["text", "source_dataset", "sentiment_std_3class", "toxic_rule_hits"]
    ].head(20)
)

print("High urgency examples:")
display(
    df[df["urgency_level"] == "high"][["text", "source_dataset", "topic_std", "urgency_rule_hits"]].head(20)
)

print("Medium urgency examples:")
display(
    df[df["urgency_level"] == "medium"][["text", "source_dataset", "topic_std", "urgency_rule_hits"]].head(20)
)

Rule-added toxic examples:


,text,source_dataset,sentiment_std_3class,toxic_rule_hits
81,giống tao rồi nó đéo đeo tai nghe đâu nói thẳn...,NEU_ESC,negative,profanity_strong
222,out đi đéo ai cấm đâu .,NEU_ESC,neutral,profanity_strong
343,thường thôi anh nhưng đéo hiểu sao trượt thể đ...,NEU_ESC,negative,profanity_strong
373,sao có thể ngu 1 cách thần kỳ thế nhỉ .,NEU_ESC,negative,insult
490,vcl lấy giấy bút đi .,NEU_ESC,neutral,internet_slang
515,hót vcl luôn .,NEU_ESC,positive,internet_slang
642,địt mẹ học ráng trọ xong tích góp liền khỏe re...,NEU_ESC,positive,profanity_strong
778,mấy thằng ngu đấy tư duy thế này sau ra cũng l...,NEU_ESC,negative,insult
831,cao hay không dương vẫn bắt bình với oanh chạy...,NEU_ESC,neutral,profanity_strong
908,không giấy ngu vlon .,NEU_ESC,negative,insult


High urgency examples:


,text,source_dataset,topic_std,urgency_rule_hits
71,trình độ ông này thì tao không quen biết nên k...,NEU_ESC,personal_affairs,violence_harassment
126,bậy bạ quan chân quang trường đại học luật hà ...,NEU_ESC,social_affairs,safety
315,xin được phép chia buồn cùng gia đình 2 chị la...,NEU_ESC,personal_affairs,safety
600,khiếp đảm hẳn là đe dọa cơ mà .,NEU_ESC,others,violence_harassment
656,nghịch lý có cầu vượt đi bộ hs sinh viên vẫn b...,NEU_ESC,social_affairs,safety
801,truong hợp lkn không van thrombectomy window ....,NEU_ESC,student_services,safety
1020,tôn căng tin quái gì anh nên đổi tên thành tôn...,NEU_ESC,personal_affairs,violence_harassment
1310,nguy hiểm bé k thuê chỗ trường chinh trường th...,NEU_ESC,student_services,safety
1456,chào mọi người mình k61 và xin phép được xưng ...,NEU_ESC,student_services,violence_harassment
1500,đăng ký hoãn thi em xin chào thầy cô và anh ch...,NEU_ESC,student_services,safety


Medium urgency examples:


,text,source_dataset,topic_std,urgency_rule_hits
58,không trường đăng kí kí túc xá muộn khả năng g...,NEU_ESC,student_services,technical_issue
78,thiết nghĩ thà bỏ luôn wifi đi cho đỡ block só...,NEU_ESC,student_services,technical_issue
160,thông báo thông tin liên quan đến thi chứng ch...,NEU_ESC,news,admin_issue
214,tham khảo trường hàng xóm khu xe mái che vip đ...,NEU_ESC,student_services,technical_issue
257,em xin hỏi một chút là lịch thi cuối kì của bọ...,NEU_ESC,academic,admin_issue
302,chào anh . thấy học a2 cũng đâu vất vậy nhỉ . ...,NEU_ESC,student_services,technical_issue
437,xuho wifi bcd tiền ít .,NEU_ESC,student_services,technical_issue
569,ốm đi khám bệnh viện châm cứu trung ương phát ...,NEU_ESC,news,admin_issue
640,điều hòa máy rượi .,NEU_ESC,others,facility_issue
667,đề nghị đại học luật hà nội dám công khai lịch...,NEU_ESC,academic,admin_issue


## 8. Export enriched dataset va report

In [9]:
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

def dist(series):
    counts = series.value_counts(dropna=False)
    return {key.item() if hasattr(key, "item") else key: int(value) for key, value in counts.items()}

report_lines = [
    "# Label Enrichment Report",
    "",
    "## Output",
    "",
    f"- Input CSV: `{INPUT_CSV.relative_to(PROJECT_ROOT)}`",
    f"- Output CSV: `{OUTPUT_CSV.relative_to(PROJECT_ROOT)}`",
    f"- Rows: `{len(df)}`",
    f"- Columns: `{len(df.columns)}`",
    "",
    "## Toxic",
    "",
    f"- Toxic distribution: `{dist(df['is_toxic'])}`",
    f"- Toxic label source: `{dist(df['toxic_label_source'])}`",
    "",
    "## Urgency",
    "",
    f"- Urgency distribution: `{dist(df['urgency_level'])}`",
    f"- Urgency label source: `{dist(df['urgency_label_source'])}`",
    "",
    "## Topic group",
    "",
    f"- Topic group distribution: `{dist(df['topic_group'])}`",
    f"- Label needs review: `{dist(df['label_needs_review'])}`",
    "",
    "## Notes",
    "",
    "- `is_toxic_seed` giu toxic goc tu Phase 1 truoc khi mo rong bang rule.",
    "- `is_toxic_rule` la nhan toxic do keyword/rule bat duoc.",
    "- `is_toxic` sau enrichment la hop cua seed va rule.",
    "- `urgency_level` la rule-based label, can review thu cong neu dung de train model nghiem tuc.",
    "- `topic_group` la taxonomy tho, khong thay the `topic_std` goc.",
]

REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print("Saved enriched CSV:", OUTPUT_CSV)
print("Saved report:", REPORT_PATH)

Saved enriched CSV: d:\AI thuc chien\Student Voice Intelligence\data\processed\student_voice_enriched.csv
Saved report: d:\AI thuc chien\Student Voice Intelligence\outputs\reports\label_enrichment_report.md
